# Stateless operators

Stateless operators are "stateless" in that they do not accumulate state. Simply put, a topology only consisting of stateless operators can swallow arbitrary many messages from the sources without ever running out of memory.


## Overview

* [map()](#map-operator)
* [peek()](#peek-operator)
* [flatmap()](#flatmap-operator)
* [filter()](#filter-operator)
* [merge()](#merge-operator)


## Preparation

Before we start off, we first prepare for the examples to follow:

In [1]:
!pip install -r ../requirements.txt

import sys
sys.path.insert(1, "../")
sys.path.insert(1, "../../..")

from kafi.streams.topologynode import TopologyNode as Tn

from generators import ClickGenerator, CustomerGenerator
click_generator = ClickGenerator()
customer_generator = CustomerGenerator()

click_source_str = "clicks"
customer_source_str = "customers"



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


Please also note that when we re-use the same example over and over again to illustrate how the operators work, we always mark the important new parts as follows:
```python
    # <------------------------------>
    ...important new parts...
    # <------------------------------>
```

<a id="map-operator"></a>
## map()

Classic `map()` operator, like e.g. in Kafka Streams.

```
map(map_fun, **kwargs)
```
* `map_fun: r -> r` the map function; gets an input record, does some processing and returns an output record.

Here is an example.

In [ ]:
built_tn = Tn.build(
    Tn.source(click_source_str)
    #
    # <------------------------------>
    .map(lambda r: {"customer_id": r["value"]["customer_id"], "view_time": r["value"]["view_time"]})
    # <------------------------------>
)

input_m_list = click_generator.generate(5)
print("Input:")
for m in input_m_list:
    print(m)

output_m_list = built_tn.process({click_source_str: input_m_list})
print("\nOutput:")
for m in output_m_list:
    print(m)


Simple. We pushed a few messages to the topology and `map()` selected two fields from them.

<a id="peek-operator"></a>
## peek()

This operator is mainly for debugging purposes. Under the covers, it's a `print` + an optional `None`-returning function that can be used to trigger side effects.

```
peek(prefix_str=None, peek_fun=None, **kwargs)
```
* `prefix_str` prefix of the printed output (if `peek_fun` is `None`)
* `peek_fun: r -> None` the `None`-returning peek function getting the input record

Here are a few examples. The first sets none of the parameters.

In [ ]:
built_tn = Tn.build(
    Tn.source(click_source_str)
    #
    # <------------------------------>
    .peek()
    # <------------------------------>
)

m_list = click_generator.generate(5)

_ = built_tn.process({click_source_str: m_list})


You can see that `peek` just printed out each of the records coming in.

Next, we use `peek` with `prefix_str` set:

In [ ]:
built_tn = Tn.build(
    Tn.source(click_source_str)
    #
    # <------------------------------>
    .peek("peek")
    # <------------------------------>
)

m_list = click_generator.generate(5)

_ = built_tn.process({click_source_str: m_list})


...and you can see that `peek()` now adds the prefix `peek: ` to the input records printed out.

Last example: We use the `peek_fun`:

In [ ]:
built_tn = Tn.build(
    Tn.source(click_source_str)
    #
    # <------------------------------>
    .peek(peek_fun=lambda r: print(f"{r}\n"))
    # <------------------------------>
)

m_list = click_generator.generate(5)

_ = built_tn.process({click_source_str: m_list})



What we did in the `peek_fun` is to add a newline after each printed out input record.

<a id="flatmap-operator"></a>
## flatmap()

This is the relational version of the classic `flatmap` operator also known from e.g. Kafka Streams. It is relational in the sense that it returns a set instead of a list of outputs.

```
flatmap(flatmap_fun, **kwargs)
```
* `flatmap_fun: r -> set(r)` the flatmap function; gets an input record and returns a set of output records.

Examples.

In [2]:
built_tn = Tn.build(
    Tn.source(customer_source_str)
    #
    # <------------------------------>
    .flatmap(lambda r: {name_part_str for name_part_str in r["value"]["name"].split(" ")})
    # <------------------------------>
)

input_m_list = customer_generator.generate(5)
print("Input:")
for m in input_m_list:
    print(m)

output_m_list = built_tn.process({customer_source_str: input_m_list})
print("\nOutput:")
for m in output_m_list:
    print(m)


Input:
{'key': '59', 'value': {'id': 59, 'name': 'Rose George'}}
{'key': '47', 'value': {'id': 47, 'name': 'Laura Randolph'}}
{'key': '76', 'value': {'id': 76, 'name': 'Jessica Nash'}}
{'key': '61', 'value': {'id': 61, 'name': 'Bobby Snyder'}}
{'key': '90', 'value': {'id': 90, 'name': 'Kimberly Walker'}}

Output:
Rose
George
Laura
Randolph
Nash
Jessica
Bobby
Snyder
Kimberly
Walker


In this example, from each customer input record, we take the `name` field of its `value`, split it and return the set of the parts of the name. So e.g., `"Alexis Jones"` becomes `{"Alexis", "Jones"}`.

A classical flatmap returns a list. In Kafi Streams, being based on a relational engine that is pydbsp, it is a set.

To see this clearly, look at the following example where we stitch together an input message where the first name is the same as the last name:

In [ ]:
built_tn = Tn.build(
    Tn.source(customer_source_str)
    #
    # <------------------------------>
    .flatmap(lambda r: {name_part_str for name_part_str in r["value"]["name"].split(" ")})
    # <------------------------------>
)

input_m_list = [{'key': '42', 'value': {'id': 42, 'name': 'Frank Frank'}}]
print("Input:")
print(input_m_list)

output_m_list = built_tn.process({customer_source_str: input_m_list})
print("Output:")
print(output_m_list)


The "philosophy" of Kafi Streams is relational, i.e., set-based, for a reason. This is not a defect but intentional.

If you miss your "classical" list-returning flatmap, don't despair - you can still recover it e.g. as below:

In [ ]:
built_tn = Tn.build(
    Tn.source(customer_source_str)
    #
    # <------------------------------>
    .flatmap(lambda r: {(i, name_part_str) for i, name_part_str in enumerate(r["value"]["name"].split(" "))})
    # <------------------------------>
)

input_m_list = [{'key': '42', 'value': {'id': 42, 'name': 'Frank Frank'}}]
print("Input:")
print(input_m_list)

output_m_list = built_tn.process({customer_source_str: input_m_list})
print("Output:")
print(output_m_list)


<a id="filter-operator"></a>
## filter()

On to the next classical stateless operator: `filter`.

```
filter(filter_fun, **kwargs)
```
* `filter_fun: r -> bool` the filter function; gets an input record and returns `True` if the record shall be kept or `False` if it shall be discarded.

Here is an example.


In [3]:
built_tn = Tn.build(
    Tn.source(click_source_str)
    #
    # <------------------------------>
    .filter(lambda r: r["value"]["view_time"] > 60)
    # <------------------------------>
)

input_m_list = click_generator.generate(5)
print("Input:")
for m in input_m_list:
    print(m)

output_m_list = built_tn.process({click_source_str: input_m_list})
print("\nOutput:")
for m in output_m_list:
    print(m)


Input:
{'key': None, 'value': {'customer_id': 12, 'view_time': 95, 'ts': 1786706850199}}
{'key': None, 'value': {'customer_id': 77, 'view_time': 29, 'ts': 1786706850299}}
{'key': None, 'value': {'customer_id': 98, 'view_time': 80, 'ts': 1786706850399}}
{'key': None, 'value': {'customer_id': 6, 'view_time': 41, 'ts': 1786706850499}}
{'key': None, 'value': {'customer_id': 2, 'view_time': 105, 'ts': 1786706850599}}

Output:
{'key': None, 'value': {'customer_id': 12, 'view_time': 95, 'ts': 1786706850199}}
{'key': None, 'value': {'customer_id': 98, 'view_time': 80, 'ts': 1786706850399}}
{'key': None, 'value': {'customer_id': 2, 'view_time': 105, 'ts': 1786706850599}}


In the example, we just keep those records where `view_time` is greater than `60`.

<a id="merge-operator"></a>
## merge()

Similar to Kafka Streams, the purpose of `merge` is to combine two branches of your topology.

```
merge(other_tn, **kwargs):
```
* `other_tn` the other topology node that the current shall be merged with

This screams for an example.


In [ ]:
click_tn = (
    Tn.source(click_source_str)
    #
    # <------------------------------>
    .map(lambda r: {"id": r["value"]["customer_id"]})
    # <------------------------------>
)

customer_tn = (
    Tn.source(customer_source_str)
    #
    # <------------------------------>
    .map(lambda r: {"id": r["value"]["id"]})
    # <------------------------------>
)

built_tn = Tn.build(
    click_tn
    # <------------------------------>
    .merge(customer_tn)
    # <------------------------------>
)

click_input_m_list = click_generator.generate(5)
print("Input (clicks):")
for m in click_input_m_list:
    print(m)

customer_input_m_list = customer_generator.generate(5)
print("\nInput (customers):")
for m in customer_input_m_list:
    print(m)

output_m_list = built_tn.process({click_source_str: click_input_m_list, customer_source_str: customer_input_m_list})
print("\nOutput:")
for m in output_m_list:
    print(m)


This is what happens:
* We create two sub topologies - one for the clicks (`click_tn`) and one for the customers (`customer_tn`).
* In each sub topology, we select just the customer ID from the input records (`customer_id` for the clicks, `id` for the customers).
* We then use the `merge()` operator to merge the outputs of the two sub topologies together.

`merge` is a stateless operation. It is neither a union or a join. Under the covers, it just adds up the weights of the ZSets of the input records:


In [ ]:
click_tn = (
    Tn.source(click_source_str)
    #
    .map(lambda r: {"id": r["value"]["customer_id"]})
)

customer_tn = (
    Tn.source(customer_source_str)
    #
    .map(lambda r: {"id": r["value"]["id"]})
)

built_tn = Tn.build(
    click_tn
    # <------------------------------>
    .merge(customer_tn)
    # <------------------------------>
)

click_input_m_list = [{'key': None, 'value': {'customer_id': 42, 'view_time': 76, 'ts': 1786618958910}}]
print("Input (clicks):")
for m in click_input_m_list:
    print(m)

customer_input_m_list = [{'key': '42', 'value': {'id': 42, 'name': 'Betty Graham MD'}}]
print("\nInput (customers):")
for m in customer_input_m_list:
    print(m)

output_m_list = built_tn.process({click_source_str: click_input_m_list, customer_source_str: customer_input_m_list})
print("\nOutput:")
for m in output_m_list:
    print(m)


In the latest example, we just sent one message to each source, both having the same customer ID (`42`). The result are *two* records, not one (under the covers, it's actually one record with weight `2`).